# Using surmise's global RNG (`set_RNG`)

surmise manages a single, user-provided random number generator for **all** of its internal sampling of random variables via `scipy.stats`.

1. User must call `surmise.set_RNG(...)` once before using any surmise functionality. Otherwise, a `RuntimeError` will be raised.
2. Any emulation/calibration workflow is reproducible, given the generator the user provides.

This notebook demonstrates these usages using the classic borehole example.

In [1]:
import numpy as np
import scipy.stats as sps

import surmise
from surmise.emulation import emulator

surmise.__version__

ModuleNotFoundError: No module named 'numpy'

## Borehole function setup
The borehole function provides a fast-to-evaluate function as a classic emulation/calibration example.  We make use of the function coded in surmise's test suite.  The only important detail here is that the function takes any parameters within the unit-cube, where $x\in [0, 1]^3, \theta \in [0, 1]^4$.

In [ ]:
from surmise.tests.shared_scenario import borehole_model

## surmise refuses to run without `set_RNG`
A fresh session has no RNG. Any surmise call that needs randomness will raise a
`RuntimeError` with an instruction.

In [ ]:
# This is to simulate a fresh session.  Generally a user does not have to clear an RNG before setting one.
from surmise._RandomNumberGenerator import RandomNumberGenerator
RandomNumberGenerator()._clear_RNG()

try:
    emulator(passthroughfunc=borehole_model, method='PCGP')
except RuntimeError as e:
    print("RuntimeError:", e)

## `set_RNG` provides one generator
Pass a `numpy.random.Generator` (i.e., `np.random.default_rng(seed)`). User may choose the same or a different generator for data generation with `scipy.stats` (via `random_state=`).  The resulting workflow will be dependent on the collection of generator(s).

In [1]:
# import secrets
# SEED = secrets.randbits(128)

SEED = 111848137687551523431846058163015350939
my_rng = np.random.default_rng(SEED)

# set RNG in surmise
surmise.set_RNG(my_rng)

DATA_SEED = 234318460581630153509391118481376875515
data_rng = np.random.default_rng(DATA_SEED)

x = sps.uniform.rvs(0, 1, size=(15, 3), random_state=data_rng)
theta = sps.uniform.rvs(0, 1, size=(50, 4), random_state=data_rng)

emu = emulator(x=x, theta=theta, f=borehole_model(x, theta), method='PCGP')
pred = emu.predict(x=x, theta=theta)
print("prediction mean shape:", pred.mean().shape)

NameError: name 'np' is not defined

Notice that `set_RNG` accepts an `np.random.Generator`.

In [2]:
for bad_rng in [np.random.RandomState(0), 12345]:
    try:
        surmise.set_RNG(bad_rng)
    except TypeError as e:
        print(f"{type(bad_rng).__name__!s:>12}: TypeError: {e}")

surmise.set_RNG(my_rng)  # restore the good one

NameError: name 'np' is not defined

## Using the same seed for reproducible results
When _all_ randomness, namely the user data generation and surmise's, are controlled via chosen seeds.  Repeating a workflow with the same collection of seeds should reproduce the results.

In [2]:
def run_workflow(seeds):
    SEED, DATA_SEED = seeds
    rng = np.random.default_rng(SEED)
    surmise.set_RNG(rng)
    data_rng = np.random.default_rng(DATA_SEED)
    x = sps.uniform.rvs(0, 1, size=(50, 3), random_state=data_rng)
    x[:, 2] = x[:, 2] > 0.5
    thetas = sps.uniform.rvs(0, 1, size=(15, 4), random_state=data_rng)
    emu = emulator(x=x, theta=thetas, f=borehole_model(x, thetas), method='PCGP')
    return emu.predict(x=x, theta=thetas).mean()

runs = {
    "m1 (SEED,   DATA)":   run_workflow((SEED,     DATA_SEED)),
    "m2 (SEED,   DATA)":   run_workflow((SEED,     DATA_SEED)),
    "m3 (SEED+1, DATA)":   run_workflow((SEED + 1, DATA_SEED)),
    "m4 (SEED,   DATA+1)": run_workflow((SEED,     DATA_SEED + 1)),
    "m5 (SEED+1, DATA+1)": run_workflow((SEED + 1, DATA_SEED + 1)),
}



In [ ]:
from itertools import combinations

for (run_name_a, run_result_a), (run_name_b, run_result_b) in combinations(runs.items(), 2):
    print(f"{run_name_a}, {run_name_b}: {np.array_equal(run_result_a, run_result_b)}")

## Changing RNG at any time
surmise keeps exactly one RNG at a time (i.e., a Python singleton).  User may replace the RNG, e.g., to restart a numerical study from a known seed.

In [ ]:
surmise.set_RNG(np.random.default_rng(SEED))
# do something
surmise.set_RNG(np.random.default_rng(SEED + 1))